<a href="https://colab.research.google.com/github/sfelan968/nfl-fantasy-predictions/blob/main/Fantasy_Football_ML_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# load dataset
player_weekly = pd.read_csv('/content/drive/MyDrive/Spring 26/BA576/BA576 Project 1/NFL-stats/weekly_player_stats_offense.csv', delimiter=',')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Spring 26/BA576/BA576 Project 1/NFL-stats/weekly_player_stats_offense.csv'

In [ ]:
player_weekly.shape

In [ ]:
player_weekly.head()

In [ ]:
# isolate all features that say "fantasy" to only keep fantasy_points_standard as y
fantasy_features = []
for feature in player_weekly.columns:
  if "fantasy" in feature:
    fantasy_features.append(feature)

fantasy_features.remove('fantasy_points_standard')

In [ ]:
# Drop all other fantasy information since fantasy_points_standard is main outcome variable
player_weekly = player_weekly.drop(fantasy_features, axis=1)

In [ ]:
player_weekly.shape

In [ ]:
# Remove unnecessary player info, since we can access players by id which is primary key
player_weekly = player_weekly.drop(['pfr_player_id'], axis=1)

In [ ]:
player_weekly.shape

**Descriptive Analytics**

In [ ]:
# Fantasy points grouped by position
position_points = player_weekly.groupby('position')['fantasy_points_standard'].mean()
position_points = position_points.sort_values(ascending=False)

position_points.plot(kind = 'bar', color = 'orange')
plt.title('Average Fantasy Points by Position')
plt.xlabel('Position')
plt.ylabel('Fantasy Points')
plt.show()

In [ ]:
# Fantasy points grouped by experience in years
points_by_year = player_weekly.groupby('years_exp')['fantasy_points_standard'].mean()

points_by_year.plot(kind = 'bar', color = 'blue')
plt.title('Average Fantasy Points by years in NFL')
plt.xlabel('Years')
plt.ylabel('Fantasy Points')
plt.show()

In [ ]:
# Years experience put into bins
# Rookie: years 1-2
# Early: years 3-5
# Mid: years 6-10
# Veteran: years 11-20
bins = [0, 2, 5, 10, 20]
labels = ['Rookie', 'Early', 'Mid', 'Veteran']
player_weekly['exp_group'] = pd.cut(player_weekly['years_exp'], bins=bins, labels=labels)

exp_points = player_weekly.groupby('exp_group')['fantasy_points_standard'].mean()
exp_points.plot(kind = 'bar', color = 'blue')

plt.title('Average Fantasy points by Stage in Career')
plt.xlabel('Years in NFL grouped by label')
plt.ylabel('Fantasy Points')
plt.show()

In [ ]:
# Histogram distribution of fantasy points
import seaborn as sns
plt.hist(player_weekly['fantasy_points_standard'], bins = 50,  density = True, alpha = 0.6)
sns.kdeplot(player_weekly['fantasy_points_standard'], color='orange', linewidth=2, fill = True, alpha= 0.3)

plt.title('Histogram distribution of fantasy points')
plt.xlabel('Fantasy points')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Correlation heatmap
# Pick only columns I want to explore since doing a heatmap on everything would be too big and too computationally expensive

# Fantasy points are heavily correlated with yards and touchdowns, makes sense
cols = [
    'fantasy_points_standard',
    'total_yards',
    'total_tds',
    'touches',

    'passing_yards',
    'pass_touchdown',
    'interception',
    'pass_attempts',

    'rushing_yards',
    'rush_touchdown',
    'rush_attempts',

    'receiving_yards',
    'receiving_touchdown',
    'targets',
    'receptions'
]

corr = player_weekly[cols].corr(numeric_only= True)
plt.figure(figsize=(20, 12))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.show()

In [ ]:
# Targets vs fantasy points
# Volume of targets doesn't gaurantee points, it averages out to a mean of around y = 20
plt.scatter(player_weekly['targets'], player_weekly['fantasy_points_standard'], alpha=0.3)
plt.xlabel('Targets')
plt.ylabel('Fantasy Points')
plt.title('Targets vs Fantasy Points')
plt.show()

In [ ]:
# Does fantasy points change by week?
# Points are relatively stable, games are much more competitive in week 18 and much less competitive in week 19
weekly_avg = player_weekly.groupby('week')['fantasy_points_standard'].mean()

weeks = weekly_avg.index
plt.xticks(np.arange(min(weeks), max(weeks) + 1, 2))
weekly_avg.plot()
plt.title('Average Fantasy Points by Week')
plt.xlabel('Week')
plt.ylabel('Fantasy Points')
plt.show()

In [ ]:
# Does team make an impact?
team_means = player_weekly.groupby('team')['fantasy_points_standard'].mean().sort_values(ascending = False)
team_means.plot(kind = "bar", color = "blue")
plt.title('Fantasy points by team')
plt.xlabel('Team')
plt.ylabel('Mean fantasy points')
plt.figure(figsize=(40, 10))
plt.show()

In [ ]:
# Any volatility in fantasy points?
# there are a few outliers
player_stats = player_weekly.groupby('player_name')['fantasy_points_standard'].agg(['mean','std'])

plt.scatter(player_stats['mean'], player_stats['std'], alpha=0.5, color = "red")
plt.xlabel('Average Points')
plt.ylabel('Volatility (Std Dev)')
plt.title('Consistency vs Performance')
plt.show()



---

X and Y & Train-Test Split

In [ ]:
model_df = player_weekly.copy()

In [ ]:
base_cols = model_df.iloc[:, 0:15].select_dtypes(include=['number']).columns.tolist()
len(base_cols)

In [ ]:
predictors = model_df.drop('fantasy_points_standard', axis=1)
rolling_cols = predictors.select_dtypes(include=['number']).drop(columns=base_cols)

bad = ["cum", "trend", "avg", "z_score"]
for col in rolling_cols.columns:
    for word in bad:
        if word in col:
            rolling_cols = rolling_cols.drop(columns=col)
            break

In [ ]:
rolling_cols = rolling_cols.columns

In [ ]:
len(rolling_cols)

In [ ]:
windows = [3, 5, 8]

for col in rolling_cols:
    for w in windows:
        model_df[f'{col}_rolling{w}'] = (
            model_df.groupby('player_id')[col]
            .transform(lambda x: x.shift(1).rolling(window=w, min_periods=1).mean())
        )

In [ ]:
rolling_feature_cols = [
    col for col in model_df.columns
    if '_rolling3' in col or '_rolling5' in col or '_rolling8' in col
]

In [ ]:
feature_cols = base_cols + rolling_feature_cols

In [ ]:
model_df['target_next_week'] = (
    model_df.groupby('player_id')['fantasy_points_standard'].shift(-1)
)

In [ ]:
model_df = model_df.dropna()

In [ ]:
model_X = model_df[feature_cols].copy()
model_y = model_df['target_next_week'].copy()

In [ ]:
latest_season = model_df['season'].max()
week_cutoff = 12

model_train_idx = (
    (model_df['season'] < latest_season) |
    ((model_df['season'] == latest_season) & (model_df['week'] < week_cutoff))
)

model_test_idx = (
    (model_df['season'] == latest_season) & (model_df['week'] >= week_cutoff)
)

X_train = model_X.loc[model_train_idx]
X_test = model_X.loc[model_test_idx]
y_train = model_y.loc[model_train_idx]
y_test = model_y.loc[model_test_idx]

In [ ]:
model_df.shape



---


Baseline Model

In [ ]:
from sklearn.metrics import root_mean_squared_error
rmse = root_mean_squared_error

player_means = (
    model_df[model_train_idx]
    .groupby('player_id')['fantasy_points_standard']
    .mean()
)

test_players = model_df[model_test_idx]['player_id']
known_mask = test_players.isin(player_means.index)

X_test = X_test[known_mask.values]
y_test = y_test[known_mask.values]

baseline_pred = (
    model_df.groupby('player_id')['fantasy_points_standard']
    .transform(lambda x: x.shift(1).expanding().mean())
)

baseline_train_preds = baseline_pred[model_train_idx]
baseline_test_preds  = baseline_pred[model_test_idx].loc[known_mask[known_mask].index]

train_valid = baseline_train_preds.notna()
test_valid  = baseline_test_preds.notna()

X_train = X_train[train_valid]
y_train = y_train[train_valid]
X_test = X_test[test_valid]
y_test = y_test[test_valid]

baseline_train_preds = baseline_train_preds[train_valid]
baseline_test_preds = baseline_test_preds[test_valid]

print(f'Baseline RMSE (train): {rmse(y_train, baseline_train_preds):.4f}')
print(f'Baseline RMSE (test):  {rmse(y_test, baseline_test_preds):.4f}')

In [ ]:
plt.scatter(x = baseline_test_preds, y = y_test, alpha=0.3, c='orange')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Perfect Prediction')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.legend()



---



In [ ]:
# =========================
# Modeling: weekly fantasy points prediction
# Goal: use prior weeks (rolling) to predict next week
# Includes: feature engineering + grouped K-fold CV + scaling/robust transforms
# =========================

from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer, RobustScaler
from sklearn.impute import SimpleImputer
df = player_weekly.copy()
# Strip whitespace from column names to prevent KeyError due to hidden characters
df.columns = df.columns.str.strip()

# --- Feature engineering ---
# Efficiency ratios
if "total_yards" in df.columns and "touches" in df.columns:
  df["yards_per_touch"] = df["total_yards"] / df["touches"].replace(0, np.nan)
if "total_tds" in df.columns and "touches" in df.columns:
  df["tds_per_touch"] = df["total_tds"] / df["touches"].replace(0, np.nan)
if "receptions" in df.columns and "targets" in df.columns:
  df["catch_rate"] = df["receptions"] / df["targets"].replace(0, np.nan)
if "receiving_yards" in df.columns and "targets" in df.columns:
  df["yards_per_target"] = df["receiving_yards"] / df["targets"].replace(0, np.nan)

# Rolling features: use week 1+2 to predict week 3 (rolling mean of prior 2 weeks)
rolling_base_cols = [
  "touches",
  "total_yards",
  "total_tds",
  "targets",
  "receptions",
  "receiving_yards",
  "rushing_yards",
  "passing_yards",
]
rolling_cols = [c for c in rolling_base_cols if c in df.columns]

if ("player_name" in df.columns) and ("season" in df.columns) and ("week" in df.columns):
  df = df.sort_values(["player_name", "season", "week"])
  for c in rolling_cols:
    df[f"{c}_roll2"] = (
      df.groupby(["player_name", "season"], sort=False)[c]\
        .transform(lambda s: s.shift(1).rolling(window=2, min_periods=1).mean())
    )

  # Lagged prior-week fantasy points (very predictive, still leakage-safe because shifted)
  if "fantasy_points_standard" in df.columns:
    df["fp_lag1"] = (
      df.groupby(["player_name", "season"], sort=False)["fantasy_points_standard"]\
        .transform(lambda s: s.shift(1))
    )
    df["fp_roll2"] = (
      df.groupby(["player_name", "season"], sort=False)["fantasy_points_standard"]\
        .transform(lambda s: s.shift(1).rolling(window=2, min_periods=1).mean())
    )

In [ ]:
# Cross validation model
# Outcome variable: fantasy points per player per match
TARGET = "fantasy_points_standard"
if TARGET not in df.columns:
  raise KeyError(f"Target column '{TARGET}' not found. Available columns: {list(df.columns)}")

y = df[TARGET]

# Predictors: player attributes + team/position + usage/performance metrics (including engineered)
X = df.drop(columns=[
  TARGET,
  "cum_fantasy_points_ppr",
  "cum_fantasy_points_half_ppr",
  "cum_fantasy_points_standard",
], errors="ignore")

# Grouped K-fold CV (k-fold variant) to reduce leakage across a player's weeks
groups = df["player_name"] if "player_name" in df.columns else None

categorical_cols = [c for c in ["position", "team", "exp_group"] if c in X.columns]
numeric_cols = [c for c in X.select_dtypes(include='number').columns if c not in categorical_cols]

numeric_pipe = Pipeline(steps=[
  ("imputer", SimpleImputer(strategy="median")),
  # Robust outlier handling + transformation (works well on heavy-tailed sports stats)
  ("quantile", QuantileTransformer(
    n_quantiles=200,
    output_distribution="normal",
    subsample=int(1e5),
    random_state=42,
  )),
  ("scaler", RobustScaler()),
])

categorical_pipe = Pipeline(steps=[
  ("imputer", SimpleImputer(strategy="most_frequent")),
  ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
  transformers=[
    ("num", numeric_pipe, numeric_cols),
    ("cat", categorical_pipe, categorical_cols),
  ],
  remainder="drop",
)

model = BaggingRegressor(
  estimator=DecisionTreeRegressor(max_depth=8, random_state=42),
  n_estimators=200,
  max_samples=0.8,
  random_state=42,
  n_jobs=-1,
)

pipe = Pipeline(steps=[
  ("preprocess", preprocess),
  ("model", model),
])

# --- Cross-validation metrics ---
cv = GroupKFold(n_splits=5) if groups is not None else 5
scoring = {"rmse": "neg_root_mean_squared_error", "r2": "r2"}
cv_results = cross_validate(
  pipe,
  X,
  y,
  cv=cv,
  groups=groups,
  scoring=scoring,
  n_jobs=-1,
  return_train_score=False,
)

rmse_scores = -cv_results["test_rmse"]
r2_scores = cv_results["test_r2"]
print("Cross-validated RMSE (mean \u00b1 std):", rmse_scores.mean(), "\u00b1", rmse_scores.std())
print("Cross-validated R2   (mean \u00b1 std):", r2_scores.mean(), "\u00b1", r2_scores.std())

# --- Optional: time-based holdout (last 20% by season/week) + plot ---
if ("season" in df.columns) and ("week" in df.columns):
  df_sorted = df.sort_values(["season", "week"])
  y_all = df_sorted[TARGET]
  X_all = df_sorted.drop(columns=[
    TARGET,
    "cum_fantasy_points_ppr",
    "cum_fantasy_points_half_ppr",
    "cum_fantasy_points_standard",
  ], errors="ignore")
else:
  X_all, y_all = X, y

split_idx = int(len(X_all) * 0.8)
X_train, X_test = X_all.iloc[:split_idx], X_all.iloc[split_idx:]
y_train, y_test = y_all.iloc[:split_idx], y_all.iloc[split_idx:]

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print("Holdout RMSE:", rmse)
print("Holdout R2:", r2)

plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Weekly Fantasy Points Prediction (Pipeline + Rolling Features)")
plt.show()

Ridge

In [ ]:
# Ridge
import pandas as pd
import numpy as np

# Reload original dataset so teammate cells do not affect this section
adi_df = player_weekly.copy()

# Basic cleaning, same idea as earlier notebook
adi_df = adi_df.dropna()

# Strip whitespace from column names
adi_df.columns = adi_df.columns.str.strip()

# Find player identifier column
player_id_col = None
for col in ['player_name', 'player_id', 'pfr_player_id']:
    if col in adi_df.columns:
        player_id_col = col
        break

if player_id_col is None:
    raise KeyError("No player identifier column found. Check your dataset columns.")

print("Using player identifier:", player_id_col)

# Sort by player and time
adi_df = adi_df.sort_values([player_id_col, 'season', 'week'])

# Create next-week target
adi_df['target_next_week'] = (
    adi_df.groupby(player_id_col)['fantasy_points_standard'].shift(-1)
)

# Drop rows where next-week target is unavailable
adi_df = adi_df.dropna(subset=['target_next_week'])

# Rolling feature columns
rolling_cols = [
    'fantasy_points_standard',
    'total_yards',
    'total_tds',
    'touches',
    'passing_yards',
    'pass_touchdown',
    'interception',
    'pass_attempts',
    'rushing_yards',
    'rush_touchdown',
    'receiving_yards',
    'receiving_touchdown',
    'targets',
    'receptions'
]

# Keep only columns that actually exist
rolling_cols = [col for col in rolling_cols if col in adi_df.columns]

# Create 3, 5, and 8 week rolling averages using only past weeks
windows = [3, 5, 8]

for col in rolling_cols:
    for w in windows:
        adi_df[f'{col}_rolling{w}'] = (
            adi_df.groupby(player_id_col)[col]
            .transform(lambda x: x.shift(1).rolling(window=w, min_periods=1).mean())
        )

# Base feature columns
base_features = [
    'years_exp',
    'week',
    'season',
    'position',
    'team',
    'exp_group'
]

# Keep only existing base features
base_features = [col for col in base_features if col in adi_df.columns]

# Use all rolling features created above
rolling_feature_cols = [
    col for col in adi_df.columns
    if '_rolling3' in col or '_rolling5' in col or '_rolling8' in col
]

adi_feature_cols = base_features + rolling_feature_cols

# Build X and y
adi_X = adi_df[adi_feature_cols].copy()
adi_y = adi_df['target_next_week'].copy()

# One-hot encode categorical variables
adi_X = pd.get_dummies(adi_X, drop_first=True)

# Time-based split:
# Train on all previous seasons + early weeks of latest season
# Test on later weeks of latest season
latest_season = adi_df['season'].max()
week_cutoff = 12

adi_train_idx = (
    (adi_df['season'] < latest_season) |
    ((adi_df['season'] == latest_season) & (adi_df['week'] < week_cutoff))
)

adi_test_idx = (
    (adi_df['season'] == latest_season) & (adi_df['week'] >= week_cutoff)
)

adi_X_train = adi_X.loc[adi_train_idx]
adi_X_test = adi_X.loc[adi_test_idx]
adi_y_train = adi_y.loc[adi_train_idx]
adi_y_test = adi_y.loc[adi_test_idx]

print("Data shape:", adi_df.shape)
print("X shape:", adi_X.shape)
print("y shape:", adi_y.shape)
print("Latest season:", latest_season)
print("Train weeks in latest season:", sorted(adi_df.loc[adi_train_idx & (adi_df['season'] == latest_season), 'week'].unique()))
print("Test weeks in latest season:", sorted(adi_df.loc[adi_test_idx, 'week'].unique()))
print("Train size:", adi_X_train.shape, adi_y_train.shape)
print("Test size:", adi_X_test.shape, adi_y_test.shape)

In [ ]:
#ridge reg
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

ridge_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('ridge', Ridge())
])

ridge_params = {
    'ridge__alpha': [0.01, 0.1, 1, 10, 100]
}

ridge_grid = GridSearchCV(
    ridge_pipeline,
    ridge_params,
    cv=5,
    scoring='neg_mean_squared_error',
    error_score='raise'
)

ridge_grid.fit(adi_X_train, adi_y_train)

ridge_best = ridge_grid.best_estimator_
ridge_preds = ridge_best.predict(adi_X_test)

ridge_rmse = np.sqrt(mean_squared_error(adi_y_test, ridge_preds))
ridge_mae = mean_absolute_error(adi_y_test, ridge_preds)
ridge_r2 = r2_score(adi_y_test, ridge_preds)

print("Best Ridge alpha:", ridge_grid.best_params_)
print("Ridge RMSE:", ridge_rmse)
print("Ridge MAE:", ridge_mae)
print("Ridge R^2:", ridge_r2)

In [ ]:
#ridge by pos
positions = ['QB', 'RB', 'WR', 'TE']

ridge_results_by_pos = []

for pos in positions:
    print(f"\n--- Position: {pos} ---")

    # Filter ds by position
    pos_idx = adi_df['position'] == pos

    pos_train_idx = adi_train_idx & pos_idx
    pos_test_idx = adi_test_idx & pos_idx

    X_train_pos = adi_X.loc[pos_train_idx].copy()
    X_test_pos = adi_X.loc[pos_test_idx].copy()
    y_train_pos = adi_y.loc[pos_train_idx]
    y_test_pos = adi_y.loc[pos_test_idx]

    print("Train size:", X_train_pos.shape[0])
    print("Test size:", X_test_pos.shape[0])

    # Remove position dummies if present
    X_train_pos = X_train_pos.drop(
        columns=[c for c in X_train_pos.columns if 'position_' in c],
        errors='ignore'
    )
    X_test_pos = X_test_pos.drop(
        columns=[c for c in X_test_pos.columns if 'position_' in c],
        errors='ignore'
    )

    # Fit Ridge for this specific position
    ridge_grid.fit(X_train_pos, y_train_pos)

    # Predictions
    train_preds = ridge_grid.predict(X_train_pos)
    test_preds = ridge_grid.predict(X_test_pos)

    # Metrics
    train_rmse = np.sqrt(mean_squared_error(y_train_pos, train_preds))
    test_rmse = np.sqrt(mean_squared_error(y_test_pos, test_preds))

    train_r2 = r2_score(y_train_pos, train_preds)
    test_r2 = r2_score(y_test_pos, test_preds)

    train_mae = mean_absolute_error(y_train_pos, train_preds)
    test_mae = mean_absolute_error(y_test_pos, test_preds)

    print("Best alpha:", ridge_grid.best_params_)
    print("Train RMSE:", train_rmse)
    print("Test RMSE:", test_rmse)
    print("Train R^2:", train_r2)
    print("Test R^2:", test_r2)

    ridge_results_by_pos.append({
        'Position': pos,
        'Best Alpha': ridge_grid.best_params_['ridge__alpha'],
        'Train RMSE': train_rmse,
        'Test RMSE': test_rmse,
        'Train R^2': train_r2,
        'Test R^2': test_r2,
        'Train MAE': train_mae,
        'Test MAE': test_mae
    })

ridge_pos_df = pd.DataFrame(ridge_results_by_pos)
ridge_pos_df

In [ ]:
# Tuning
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Pipeline
ridge_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('ridge', Ridge())
])

# Wide alpha grid (log scale)
ridge_params = {
    'ridge__alpha': np.logspace(-4, 5, 100)
}

# Grid search with CV
ridge_grid = GridSearchCV(
    ridge_pipeline,
    ridge_params,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

# Fit model
ridge_grid.fit(adi_X_train, adi_y_train)

# Best model + predictions
ridge_best = ridge_grid.best_estimator_
ridge_preds = ridge_best.predict(adi_X_test)

# Metrics
ridge_rmse = np.sqrt(mean_squared_error(adi_y_test, ridge_preds))
ridge_mae = mean_absolute_error(adi_y_test, ridge_preds)
ridge_r2 = r2_score(adi_y_test, ridge_preds)

print("Best Ridge alpha:", ridge_grid.best_params_)
print("Ridge RMSE:", ridge_rmse)
print("Ridge MAE:", ridge_mae)
print("Ridge R^2:", ridge_r2)

# ==============================
# Plot: Alpha vs RMSE (for presentation)
# ==============================

ridge_results = pd.DataFrame(ridge_grid.cv_results_)
ridge_results['RMSE'] = np.sqrt(-ridge_results['mean_test_score'])

plt.figure(figsize=(8,5))
plt.plot(ridge_results['param_ridge__alpha'], ridge_results['RMSE'], marker='o')
plt.xscale('log')
plt.xlabel('Alpha (Regularization Strength)')
plt.ylabel('Cross-Validated RMSE')
plt.title('Ridge Regression Hyperparameter Tuning')
plt.grid(True)
plt.show()



---

Lasso Model

In [ ]:
# finding best alpha for lasso model
from sklearn.linear_model import LassoCV

lasso_cv = LassoCV(alphas=np.logspace(-4, 2, 50), cv=5)
lasso_cv.fit(X_train, y_train)
best_alpha = lasso_cv.alpha_

In [ ]:
print("Best alpha:", lasso_cv.alpha_)

In [ ]:
# lasso with best predictors/alpha
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import Lasso
from sklearn.pipeline import make_pipeline
from sklearn.metrics import root_mean_squared_error

rmse = root_mean_squared_error

scaler = StandardScaler()
lasso_pipe = make_pipeline(scaler, SelectFromModel(Lasso(alpha=best_alpha)), Lasso(alpha=best_alpha))
lasso_pipe.fit(X_train, y_train)
y_pred = lasso_pipe.predict(X_test)

print(f'RMSE on X_train with best predictors (alpha = {best_alpha:0.4f}): {rmse(lasso_pipe.predict(X_train), y_train)}')
print(f'RMSE on X_test with best predictors (alpha = {best_alpha:0.4f}): {rmse(y_pred, y_test)}')
print(f'RMSE of baseline model: {rmse(baseline_test_preds, y_test)}')

In [ ]:
 # absolute difference in RMSE from baseline to lasso model

diff = np.abs(rmse(baseline_test_preds, y_test) - rmse(y_pred, y_test))
print(f'Absolute difference in RMSE from baseline model to fitted lasso model:', diff)

In [ ]:
# visualize results
plt.scatter(x = baseline_test_preds, y = y_test, alpha=0.3, c='orange', label='Baseline Model')
plt.scatter(x = y_pred, y = y_test, alpha=0.3, c='cornflowerblue', label='Lasso Model')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.legend()

In [ ]:
selector = lasso_pipe.named_steps['selectfrommodel']
mask = selector.get_support()

feature_names = np.array(X_train.columns)[mask]
coefficients = selector.estimator_.coef_[mask]

importance_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients
}).sort_values('coefficient', key=abs, ascending=False)

print(importance_df[0:5])

In [ ]:
len(feature_names)

In [ ]:
importance_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients
}).sort_values('coefficient', key=abs, ascending=False).head(10)

plt.figure(figsize=(10, len(importance_df) * 0.4))
plt.barh(importance_df['feature'], importance_df['coefficient'])
plt.axvline(x=0, color='black', linewidth=0.8)
plt.xlabel('Coefficient')
plt.title('Lasso Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
selected = lasso_pipe[1].get_support()
feature_names = X_train.columns[selected]
coefs = lasso_pipe[2].coef_
print(sorted(zip(feature_names, coefs), key=lambda x: abs(x[1]), reverse=True)[:10])

In [ ]:
positions = ['QB', 'RB', 'WR', 'TE']
results = {}

for pos in positions:
    pos_mask = model_df['position'] == pos
    pos_data = model_df[pos_mask]

    X_pos = model_X[pos_mask]
    y_pos = model_y[pos_mask]

    train_idx_pos = model_train_idx[pos_mask]
    test_idx_pos  = model_test_idx[pos_mask]

    X_train_pos = X_pos[train_idx_pos]
    X_test_pos  = X_pos[test_idx_pos]
    y_train_pos = y_pos[train_idx_pos]
    y_test_pos  = y_pos[test_idx_pos]

    known_mask_pos = pos_data[test_idx_pos]['player_id'].isin(
        pos_data[train_idx_pos].groupby('player_id')['fantasy_points_standard'].mean().index
    )

    X_test_pos = X_test_pos.loc[known_mask_pos[known_mask_pos].index]
    y_test_pos = y_test_pos.loc[known_mask_pos[known_mask_pos].index]

    baseline_pred_pos = (
        pos_data.groupby('player_id')['fantasy_points_standard']
        .transform(lambda x: x.shift(1).expanding().mean())
    )
    baseline_train_pos = baseline_pred_pos[train_idx_pos]
    baseline_test_pos  = baseline_pred_pos[test_idx_pos].loc[known_mask_pos[known_mask_pos].index]

    train_valid_pos = baseline_train_pos.notna()
    test_valid_pos  = baseline_test_pos.notna()

    lasso_cv_pos = LassoCV(alphas=np.logspace(-4, 2, 50), cv=5)
    lasso_cv_pos.fit(X_train_pos, y_train_pos)

    lasso_pipe_pos = make_pipeline(
        StandardScaler(),
        SelectFromModel(Lasso(alpha=lasso_cv_pos.alpha_)),
        Lasso(alpha=lasso_cv_pos.alpha_)
    )
    lasso_pipe_pos.fit(X_train_pos, y_train_pos)

    results[pos] = {
        'lasso_train_rmse':    rmse(y_train_pos[train_valid_pos], lasso_pipe_pos.predict(X_train_pos[train_valid_pos])),
        'lasso_test_rmse':     rmse(y_test_pos[test_valid_pos],   lasso_pipe_pos.predict(X_test_pos[test_valid_pos])),
        'baseline_train_rmse': rmse(y_train_pos[train_valid_pos], baseline_train_pos[train_valid_pos]),
        'baseline_test_rmse':  rmse(y_test_pos[test_valid_pos],   baseline_test_pos[test_valid_pos]),
        'alpha': lasso_cv_pos.alpha_
    }

In [ ]:
for pos, res in results.items():
    print(f"{pos}: Lasso=({res['lasso_train_rmse']:.4f}, {res['lasso_test_rmse']:.4f}), Baseline=({res['baseline_train_rmse']:.4f}, {res['baseline_test_rmse']:.4f}), alpha={res['alpha']:.4f}")

Random Forest


In [ ]:
# Random forest preprocessing

# imports
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# establish target and vectorize position
TARGET = 'fantasy_points_standard'
le = LabelEncoder()
player_weekly['position_enc'] = le.fit_transform(player_weekly['position'].astype(str))

In [ ]:
# Pipeline to evaluate random forest on all possibly important metrics
def evaluate(name, model, X_train, X_test, y_train, y_test, features):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse  = np.sqrt(mean_squared_error(y_test, preds))
    mae   = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    n, p  = len(y_test), len(features)
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)


    # Training metrics
    train_preds = model.predict(X_train)
    train_rmse  = np.sqrt(mean_squared_error(y_train, train_preds))
    train_r2    = r2_score(y_train, train_preds)

    imp = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)

    # Print metrics
    print(f"{name}")
    print(f"Features used: {len(features)}")
    print(f"\n               {'Train':>10} {'Test':>10}")
    print(f"  RMSE       : {train_rmse:>10.4f} {rmse:>10.4f}")
    print(f"  R²         : {train_r2:>10.4f} {r2:>10.4f}")
    print(f"  Adj. R²    : {'N/A':>10} {adj_r2:>10.4f}")
    print(f"  Overfit gap: {train_r2 - r2:>10.4f}")
    print(f"\n  Top 10 Feature Importances:")
    for feat, val in imp.head(10).items():
        print(f"    {feat:<45} {val:.4f}")

    return {"model": model, "rmse": rmse, "mae": mae, "Adjusted r squared": adj_r2, "importances": imp}


In [ ]:
# Model 1: use passing, receiving, rushing as features

features_1 = [
    # Passing
    'pass_attempts', 'complete_pass', 'passing_yards', 'pass_touchdown',
    'interception', 'comp_pct', 'ypa', 'passer_rating', 'passing_air_yards',
    'times_sacked', 'times_pressured',
    # Receiving
    'targets', 'receptions', 'receiving_yards', 'receiving_touchdown',
    'yards_after_catch', 'yptarget', 'ypr', 'rec_adot', 'receiving_air_yards',
    # Rushing
    'rush_attempts', 'rushing_yards', 'rush_touchdown', 'ypc',
    'rush_attempts_red_zone', 'rush_touchdown_red_zone',
    # General offense
    'touches', 'total_yards', 'total_tds', 'td_pct', 'yptouch',
    'first_down_pass', 'first_down_rush',
    'third_down_converted', 'fourth_down_converted',
    'offense_snaps', 'offense_pct',
    # Context
    'position_enc', 'week', 'season', 'years_exp',
]

player_weekly1 = player_weekly[features_1 + [TARGET]].dropna()
# establish that X is the features and Y is fantasy_points_standard
X = player_weekly1[features_1]
y = player_weekly1[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 42, test_size = 0.2)

random_forest = RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_leaf=5,
                             max_features='sqrt', random_state=42, n_jobs=-1)

evaluation1 = evaluate("Model 1: Raw Weekly Box-Score Stats", random_forest, X_train, X_test, y_train, y_test, features_1)


# adjusted r^2 with defaultparameters is 0.9460

In [ ]:
features_2 = [
    # Season averages
    'avg_passing_yards', 'avg_pass_touchdown',
    'avg_rushing_yards', 'avg_rush_touchdown', 'avg_receiving_yards',
    'avg_targets', 'avg_receptions', 'avg_receiving_touchdown',
    'avg_touches', 'avg_total_yards', 'avg_total_tds',
    'avg_comp_pct', 'avg_ypa', 'avg_ypc', 'avg_yptarget',
    # 4-week rolling averages
    'prev_4_avg_passing_yards',
    'prev_4_avg_rush_attempts', 'prev_4_avg_rushing_yards',
    'prev_4_avg_targets', 'prev_4_avg_receptions', 'prev_4_avg_receiving_yards',
    'prev_4_avg_touches', 'prev_4_avg_total_yards', 'prev_4_avg_total_tds',
    # Season-long trends (slope)
    'trend_passing_yards', 'trend_rushing_yards', 'trend_receiving_yards',
    'trend_targets', 'trend_total_tds', 'trend_total_yards',
    # 4-week trends
    'prev_4_trend_passing_yards', 'prev_4_trend_rushing_yards',
    'prev_4_trend_receiving_yards', 'prev_4_trend_total_tds',
    # Z-scores
    'z_score_passing_yards', 'z_score_rushing_yards',
    'z_score_receiving_yards', 'z_score_total_yards', 'z_score_total_tds',
    # Z-scores vs prev 4 weeks
    'z_score_prev_4_passing_yards', 'z_score_prev_4_rushing_yards',
    'z_score_prev_4_receiving_yards',
    # Context
    'position_enc', 'week', 'season', 'years_exp', 'weight', 'height',
]

player_weekly2 = player_weekly[features_2 + [TARGET]].dropna()
# establish that X is the features and Y is fantasy_points_standard
X = player_weekly2[features_2]
y = player_weekly2[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 42, test_size = 0.2)

random_forest = RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_leaf=5,
                             max_features='sqrt', random_state=42, n_jobs=-1)

evaluation2 = evaluate("Model 2: averages", random_forest, X_train, X_test, y_train, y_test, features_2)
# adjusted r^2 with defaultparameters is 0.9004

In [ ]:
# adjust hyperparameters with model 3 with RandommizedSearch CV

from sklearn.model_selection import RandomizedSearchCV
param_dist = {
    'n_estimators':     [100, 200, 300, 500],
    'max_depth':        [5, 10, 15, 20, None],
    'min_samples_leaf': [1, 3, 5, 10],
    'max_features':     ['sqrt', 'log2', 1.0],
}

best_features = ['z_score_total_tds', 'avg_total_tds', 'avg_total_yards',
'total_tds', 'td_pct', 'total_yards']

player_weekly3 = player_weekly[best_features + [TARGET]].dropna()
# establish that X is the features and Y is fantasy_points_standard
X = player_weekly3[best_features]
y = player_weekly3[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 42, test_size = 0.2)

# Do randomized search cv 5 folds with 20 iteration
random_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_dist,
    n_iter=20,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=1
)
random_search.fit(X_train, y_train)
print(random_search.best_params_)

In [ ]:
# model 3: try the top 3 features from each of the first two models
best_features = ['z_score_total_tds', 'avg_total_tds', 'avg_total_yards',
'total_tds', 'td_pct', 'total_yards']

player_weekly3 = player_weekly[best_features + [TARGET]].dropna()
# establish that X is the features and Y is fantasy_points_standard
X = player_weekly3[best_features]
y = player_weekly3[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 42, test_size = 0.2)

random_forest = RandomForestRegressor(n_estimators=300, max_depth=10, min_samples_leaf=3,
                             max_features=1.0, random_state=42, n_jobs=-1)

evaluation1 = evaluate("Model 3: Top 3 features from first two models", random_forest, X_train, X_test, y_train, y_test, best_features)
# adjusted r^2 with defaultparameters is 0.9213
# adjusted r^2 with adjusted hyperparameters is 0.9227

In [ ]:
# Train the 6 features by NFL skill position (Quarterback, Runningback, Wide Receiver, Tight End)

best_features = ['z_score_total_tds', 'avg_total_tds', 'avg_total_yards',
                 'total_tds', 'td_pct', 'total_yards']

for pos in ['QB', 'RB', 'WR', 'TE']:
    pos_data = player_weekly[player_weekly['position'] == pos][best_features + [TARGET]].dropna()
    X = pos_data[best_features]
    y = pos_data[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    rf = RandomForestRegressor(n_estimators=300, max_depth=10, min_samples_leaf=3,
                               max_features=1.0, random_state=42, n_jobs=-1)
    evaluate(f"{pos} MODEL", rf, X_train, X_test, y_train, y_test, best_features)

## Model Generalization & Results Summary


### Generalization Discussion
Model 1 achieves the highest test R² of (0.946) and a small overfitting gap of (~0.024),
indicating strong generalization. This makes sense — raw box-score stats like total
yards and touchdowns directly determine fantasy points, so the signal is clean and
consistent across train and test sets.

Model 2 (rolling averages and z-scores) generalizes slightly worse (test R² ~0.900,
overfit gap ~0.060). These engineered features capture player trends and relative
performance, which are more variable week-to-week and introduce more noise,
making the model slightly harder to generalize.

The first two models were to get a general sense of what features to use for my third model.

Model 3 uses only the top 3 features from
Models 1 and 2, making 6 features in total for model 3 . Despite its simplicity, it achieves a test R² of 0.923, which is nearly
matching Model 1 with a small overfit gap (~0.027). This is a strong result:
fewer features, similar performance, and comparable generalization. It suggests
these 6 features (total_tds, total_yards, td_pct, and their rolling/z-score
variants) capture most of the predictive signal with less risk of overfitting.

### Limitations
- **Data leakage risk in Model 1**: Raw same-game stats (e.g. total_yards in week N
  predicting fantasy points in week N) are computed from the same game as the
  outcome. In a real deployment you would only have *prior* week data available,
  making Model 2/3 more realistic for actual fantasy predictions.
- **Positional heterogeneity**: QBs, RBs, WRs, and TEs score fantasy points through
  completely different mechanisms. The position-stratified models confirm this —
  a single model trained on all positions must learn these differences implicitly,
  which may limit ceiling performance.
- **Generalization across seasons**: The model is trained and tested on shuffled
  weekly data, not split by season. Performance on a truly held-out future season
  may be lower than the test R² suggests.

Boosting

In [ ]:
# Boosting

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

boost_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('boost', GradientBoostingRegressor(random_state=42))
])

boost_params = {
    'boost__n_estimators': [100, 200],
    'boost__learning_rate': [0.05, 0.1],
    'boost__max_depth': [2, 3, 4],
    'boost__subsample': [0.7, 0.9]
}

boost_grid = GridSearchCV(
    boost_pipeline,
    boost_params,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

boost_grid.fit(X_train, y_train)

boost_best = boost_grid.best_estimator_
boost_preds = boost_best.predict(X_test)

boost_rmse = np.sqrt(mean_squared_error(y_test, boost_preds))
boost_mae = mean_absolute_error(y_test, boost_preds)
boost_r2 = r2_score(y_test, boost_preds)

print("Best Boosting params:", boost_grid.best_params_)
print("Boosting RMSE:", boost_rmse)
print("Boosting MAE:", boost_mae)
print("Boosting R^2:", boost_r2)


In [ ]:
# Evan's boosting model
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

TARGET = 'fantasy_points_standard'

features_boost = [
    # Passing
    'pass_attempts', 'complete_pass', 'passing_yards', 'pass_touchdown',
    'interception', 'comp_pct', 'ypa', 'passer_rating', 'passing_air_yards',
    'times_sacked', 'times_pressured',
    # Receiving
    'targets', 'receptions', 'receiving_yards', 'receiving_touchdown',
    'yards_after_catch', 'yptarget', 'ypr', 'rec_adot', 'receiving_air_yards',
    # Rushing
    'rush_attempts', 'rushing_yards', 'rush_touchdown', 'ypc',
    'rush_attempts_red_zone', 'rush_touchdown_red_zone',
    # General offense
    'touches', 'total_yards', 'total_tds', 'td_pct', 'yptouch',
    'first_down_pass', 'first_down_rush',
    'third_down_converted', 'fourth_down_converted',
    'offense_snaps', 'offense_pct',
    # Context
    'week', 'season', 'years_exp',
]

boost_data = player_weekly[features_boost + [TARGET]].dropna()
X = boost_data[features_boost]
y = boost_data[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

boost_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('boost', XGBRegressor(random_state=42, n_jobs=-1))
])

boost_params = {
    'boost__n_estimators': [200, 500],
    'boost__learning_rate': [0.05, 0.1],
    'boost__max_depth': [3, 6],
    'boost__subsample': [0.8, 1.0],
}

boost_grid = GridSearchCV(
    boost_pipeline,
    boost_params,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

boost_grid.fit(X_train, y_train)

boost_best = boost_grid.best_estimator_

# Test metrics
boost_preds = boost_best.predict(X_test)
boost_rmse = np.sqrt(mean_squared_error(y_test, boost_preds))
boost_r2 = r2_score(y_test, boost_preds)

# Train metrics
train_preds = boost_best.predict(X_train)
train_rmse = np.sqrt(mean_squared_error(y_train, train_preds))
train_r2 = r2_score(y_train, train_preds)

print("Best params:", boost_grid.best_params_)
print(f"\n               {'Train':>10} {'Test':>10}")
print(f"  RMSE       : {train_rmse:>10.4f} {boost_rmse:>10.4f}")
print(f"  R²         : {train_r2:>10.4f} {boost_r2:>10.4f}")
print(f"  Overfit gap: {train_r2 - boost_r2:>10.4f}")

In [ ]:
# Bagging

from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

bagging_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('bagging', BaggingRegressor(
        estimator=DecisionTreeRegressor(random_state=42),
        random_state=42,
        n_jobs=-1
    ))
])

bagging_params = {
    'bagging__n_estimators': [100, 200],
    'bagging__max_samples': [0.7, 0.9],
    'bagging__estimator__max_depth': [5, 10, None],
    'bagging__estimator__min_samples_leaf': [1, 3, 5]
}

bagging_grid = GridSearchCV(
    bagging_pipeline,
    bagging_params,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

bagging_grid.fit(X_train, y_train)

bagging_best = bagging_grid.best_estimator_
bagging_preds = bagging_best.predict(X_test)

bagging_rmse = np.sqrt(mean_squared_error(y_test, bagging_preds))
bagging_mae = mean_absolute_error(y_test, bagging_preds)
bagging_r2 = r2_score(y_test, bagging_preds)

print("Best Bagging params:", bagging_grid.best_params_)
print("Bagging RMSE:", bagging_rmse)
print("Bagging MAE:", bagging_mae)
print("Bagging R^2:", bagging_r2)

In [ ]:
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

TARGET = 'fantasy_points_standard'

features_boost = [
    # Passing
    'pass_attempts', 'complete_pass', 'passing_yards', 'pass_touchdown',
    'interception', 'comp_pct', 'ypa', 'passer_rating', 'passing_air_yards',
    'times_sacked', 'times_pressured',
    # Receiving
    'targets', 'receptions', 'receiving_yards', 'receiving_touchdown',
    'yards_after_catch', 'yptarget', 'ypr', 'rec_adot', 'receiving_air_yards',
    # Rushing
    'rush_attempts', 'rushing_yards', 'rush_touchdown', 'ypc',
    'rush_attempts_red_zone', 'rush_touchdown_red_zone',
    # General offense
    'touches', 'total_yards', 'total_tds', 'td_pct', 'yptouch',
    'first_down_pass', 'first_down_rush',
    'third_down_converted', 'fourth_down_converted',
    'offense_snaps', 'offense_pct',
    # Context
    'week', 'season', 'years_exp',
]

boost_data = player_weekly[features_boost + [TARGET]].dropna()
X = boost_data[features_boost]
y = boost_data[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

boost_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('boost', XGBRegressor(random_state=42, n_jobs=-1))
])

boost_params = {
    'boost__n_estimators': [100, 200, 300, 500],
    'boost__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'boost__max_depth': [2, 3, 4, 6],
    'boost__subsample': [0.7, 0.8, 0.9, 1.0],
}

boost_grid = GridSearchCV(
    boost_pipeline,
    boost_params,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

boost_grid.fit(X_train, y_train)

boost_best = boost_grid.best_estimator_

# Test metrics
boost_preds = boost_best.predict(X_test)
boost_rmse = np.sqrt(mean_squared_error(y_test, boost_preds))
boost_r2 = r2_score(y_test, boost_preds)

# Train metrics
train_preds = boost_best.predict(X_train)
train_rmse = np.sqrt(mean_squared_error(y_train, train_preds))
train_r2 = r2_score(y_train, train_preds)

print("Best params:", boost_grid.best_params_)
print(f"\n               {'Train':>10} {'Test':>10}")
print(f"  RMSE       : {train_rmse:>10.4f} {boost_rmse:>10.4f}")
print(f"  R²         : {train_r2:>10.4f} {boost_r2:>10.4f}")
print(f"  Overfit gap: {train_r2 - boost_r2:>10.4f}")

In [ ]:
# ============================================================
# K-Fold Cross Validation Using Existing Models
# ============================================================

from sklearn.model_selection import KFold, cross_validate
from sklearn.base import clone
import pandas as pd

# 5-fold CV setup
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Scoring metrics
scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}

# Use the same X and y you already created
# X = model features
# y = target variable

existing_models = {
    "Baseline Model": baseline_test_preds,
    "Ridge Regression": ridge_best,
    "Lasso Regression": lasso_pipeline,
    "Random Forest": random_forest,
    "Bagging": bagging_best,
    "Boosting": boost_best
}

cv_results = []

for name, existing_model in existing_models.items():

    scores = cross_validate(
        clone(existing_model),   # uses your existing model structure
        X,
        y,
        cv=kf,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "Train RMSE": -scores["train_rmse"].mean(),
        "Test RMSE": -scores["test_rmse"].mean(),
        "RMSE Std Dev": scores["test_rmse"].std(),
        "Test MAE": -scores["test_mae"].mean(),
        "Test R²": scores["test_r2"].mean()
    })

results_df = pd.DataFrame(cv_results)
results_df = results_df.sort_values(by="Test RMSE")

print(results_df)

Neural Network


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
from sklearn.model_selection import train_test_split

# ── 1. Prep features (Model 3 top features) ──
features_nn = ['z_score_total_tds', 'avg_total_tds', 'avg_total_yards',
               'total_tds', 'td_pct', 'total_yards']

TARGET = 'fantasy_points_standard'

nn_data = player_weekly[features_nn + [TARGET]].dropna()
X_nn = nn_data[features_nn].values
y_nn = nn_data[TARGET].values

X_train_nn, X_test_nn, y_train_nn, y_test_nn = train_test_split(
    X_nn, y_nn, test_size=0.2, random_state=42
)

# Neural networks are sensitive to scale, so standardize
scaler = StandardScaler()
X_train_nn = scaler.fit_transform(X_train_nn)
X_test_nn  = scaler.transform(X_test_nn)

# ── 2. Convert to tensors ──
X_train_t = torch.tensor(X_train_nn, dtype=torch.float32)
y_train_t = torch.tensor(y_train_nn, dtype=torch.float32).unsqueeze(1)
X_test_t  = torch.tensor(X_test_nn,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test_nn,  dtype=torch.float32).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)

# ── 3. Define the network ──
class FantasyNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_nn = FantasyNet(input_dim=len(features_nn)).to(device)

# ── 4. Train ──
optimizer = torch.optim.Adam(model_nn.parameters(), lr=1e-3)
loss_fn   = nn.MSELoss()

EPOCHS = 50
train_losses = []

for epoch in range(EPOCHS):
    model_nn.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model_nn(X_batch)
        loss  = loss_fn(preds, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_loader))
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {train_losses[-1]:.4f}")

# ── 5. Evaluate ──
model_nn.eval()
with torch.no_grad():
    train_preds = model_nn(X_train_t.to(device)).cpu().numpy()
    test_preds  = model_nn(X_test_t.to(device)).cpu().numpy()

train_r2   = r2_score(y_train_nn, train_preds)
test_r2    = r2_score(y_test_nn,  test_preds)
train_rmse = np.sqrt(mean_squared_error(y_train_nn, train_preds))
test_rmse  = np.sqrt(mean_squared_error(y_test_nn,  test_preds))

print("\nNeural Network (Model 3 Features)")
print(f"               {'Train':>10} {'Test':>10}")
print(f"  RMSE       : {train_rmse:>10.4f} {test_rmse:>10.4f}")
print(f"  R²         : {train_r2:>10.4f} {test_r2:>10.4f}")
print(f"  Overfit gap: {train_r2 - test_r2:>10.4f}")

# ── 6. Training loss curve ──
plt.plot(train_losses)
plt.title('Neural Network Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.show()

# ── 7. Actual vs Predicted ──
plt.scatter(y_test_nn, test_preds, alpha=0.3)
plt.xlabel('Actual Fantasy Points')
plt.ylabel('Predicted Fantasy Points')
plt.title('Neural Network: Actual vs Predicted')
plt.plot([y_test_nn.min(), y_test_nn.max()],
         [y_test_nn.min(), y_test_nn.max()], 'r--')
plt.show()

## Neural Network Results ##

Epoch 10/50 — Loss: 6.8374
Epoch 20/50 — Loss: 6.2167
Epoch 30/50 — Loss: 5.6900
Epoch 40/50 — Loss: 5.4786
Epoch 50/50 — Loss: 5.3780

Neural Network Metrics (Model 3 Features)
                           
  RMSE       :    Train 2.6082    Test  2.6212

  R²         :    Train 0.8899   Test  0.8888

  Overfit gap:     0.0011